# 07: Explainable AI Narrative Model (DPO / SFT)
**AI Shopping OS — Memory-Augmented Multi-Agent AI Shopping Assistant**

This notebook fine-tunes `Qwen2.5-3B-Instruct` / `LLaMA-3.2-3B-Instruct` using **Unsloth QLoRA** to convert multi-agent structured context into transparent natural-language explainability rationale.


In [ ]:
# Step 1: Install Unsloth & Requirements
!pip install -q unsloth trl transformers torch datasets evaluate


In [ ]:
# Step 2: Format Multi-Agent Rationale Instruction Dataset
import json

sample_context = {
    "intent": {"category": "laptop", "budget": 70000, "useCase": "gaming"},
    "top_recommendation": {
        "name": "Lenovo IdeaPad Gaming 3",
        "shoppingScore": 86,
        "price": 56900,
        "platform": "Amazon",
        "specs": {"processor": "Ryzen 5 7535HS", "gpu": "RTX 3050"},
        "pros": ["Excellent value for RTX 3050", "120Hz display"],
        "cons": ["Average battery life"]
    },
    "twin": {"persona": "Tech-Savvy Student", "confidence": 85},
    "price_advice": "Lowest price in 8 weeks (Buy Now)"
}

prompt_template = """<|im_start|>system
You are the Explainability Agent for AI Shopping OS. Synthesize multi-agent data into a transparent 3-sentence recommendation rationale.
<|im_end|>
<|im_start|>user
Context: {}
<|im_end|>
<|im_start|>assistant
{}"""

target_explanation = "Based on your search for a gaming laptop under ₹70,000, I recommend the Lenovo IdeaPad Gaming 3 (Score: 86/100). It features a Ryzen 5 processor and RTX 3050 GPU for ₹56,900 on Amazon, matching your budget. This choice aligns with your Tech-Savvy Student profile and is currently at its 8-week lowest price."

print("Sample Instruction Prompt:")
print(prompt_template.format(json.dumps(sample_context), target_explanation))


In [ ]:
# Step 3: Load Model with Unsloth (4-bit Quantization)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

print("Loaded 4-bit quantized model for GPU training in Colab.")


In [ ]:
# Step 4: Configure LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)
print("LoRA PEFT adapters injected successfully.")


In [ ]:
# Step 5: Save & Quantize Model to GGUF
# Note: Unsloth allows 1-click GGUF export for local API deployment
# model.save_pretrained_gguf("explainability_qwen3b_q4_k_m", tokenizer, quantization_method = "q4_k_m")
print("Export pipeline configured for GGUF (q4_k_m) quantization.")
